# 04 — Classificazione Lamfalussy a 4 Livelli + Heatmap di Ibridità

Questo notebook sostituisce la pipeline a layer emergenti con la tassonomia canonica
**Lamfalussy a 4 livelli** (Commissione Europea, 2001 → riforma post-crisi 2010).

| Fase | Input | Operazione | Output |
|---|---|---|---|
| **A** | Testo articolo + full doc | LLM: classifica provision per provision in L1–L4 | `segments_lamfalussy.csv` |
| **B** | Classificazioni per articolo | Calcolo entropia normalizzata per articolo | `nodes_lamfalussy.csv` |
| **C** | Entropia per articolo | Aggregazione → score ibridità per atto | `nodes_hybridity.csv` |
| **D** | CSV di output | Build `heatmaps.json` + patch HTML | file aggiornati |

## 0. Configurazione

**Modifica solo questa cella.**

In [2]:
MATERIA_NAME = "appalti_it"

# ── Modello ────────────────────────────────────────────────────────────────────
LLM_MODEL          = "gpt-4.1-mini"   # modello da usare per la classificazione

# ── Parametri API ─────────────────────────────────────────────────────────────
LLM_MAX_TOKENS     = 2000   # token risposta (JSON con provisions può essere lungo)
LLM_DELAY_SECONDS  = 0.3
LLM_MAX_RETRIES    = 3
LLM_RETRY_DELAY    = 5.0

# ── Parallelismo ──────────────────────────────────────────────────────────────
MAX_WORKERS        = 5      # thread paralleli per le chiamate API

# ── Checkpoint ────────────────────────────────────────────────────────────────
CHECKPOINT_EVERY   = 50     # articoli tra un salvataggio e il successivo

# ── Contesto documento (cap per non sforare il context window) ────────────────
DOC_CONTEXT_MAX_CHARS = 40_000   # caratteri massimi del documento di contesto

## 1. Import e Percorsi

In [3]:
import os, re, json, math, time
import numpy as np
import pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
from dotenv import load_dotenv
load_dotenv(dotenv_path=r'C:\Users\claud\Documents\GitHub\eu-law-network-viz\.env')

from openai import OpenAI
import openai

# ── Percorsi ──────────────────────────────────────────────────────────────────
output_path = os.path.join('..', 'data', 'output', MATERIA_NAME)
Path(output_path).mkdir(parents=True, exist_ok=True)

INPUT_FILE              = os.path.join(output_path, 'nodes_texts_it.csv')
INPUT_FILE_EU           = os.path.join(output_path, 'nodes_texts_eu_appalti.csv')  # direttive EU da 00
SEGMENTS_LAMF_FILE      = os.path.join(output_path, 'segments_lamfalussy.csv')
SEGMENTS_LAMF_CKPT_FILE = os.path.join(output_path, 'segments_lamfalussy_checkpoint.csv')
NODES_LAMFALUSSY_FILE   = os.path.join(output_path, 'nodes_lamfalussy.csv')
NODES_HYBRIDITY_FILE    = os.path.join(output_path, 'nodes_hybridity.csv')

# HTML da patchare (relativo a questo notebook)
HTML_FILE = os.path.join('..', 'procurements_network.html')

# Prompt template esterno — modifica questo file per cambiare il prompt
PROMPT_FILE = os.path.join('..', 'notebooks', 'prompt.txt')

# ── Costanti Lamfalussy ────────────────────────────────────────────────────────
LAMFALUSSY_LEVELS = [
    {
        'key':  'L1',
        'name': 'Level 1 — Framework principles',
        'label': 'level_1',
        'description': (
            'Level 1 legislation sets out the core framework principles and defines essential features, '
            'as adopted by the European Parliament and Council in the co-decision procedure. '
            'Level 1 includes, for example, the objectives of the regulation; its scope and key definitions; '
            'the main rights and obligations; the institutional framework; core choices regarding harmonization '
            'between national and European levels; and the delegation clauses empowering the Commission or '
            'regulatory agencies to adopt implementing measures. These elements reflect fundamental political '
            'choices and are intended to be stable and enduring.'
        ),
    },
    {
        'key':  'L2',
        'name': 'Level 2 — Operational rules',
        'label': 'level_2',
        'description': (
            'Level 2 legislation translates Level 1 principles into operational rules by specifying their '
            'technical content and modes of application, adopted by the Commission via delegated acts, '
            'implementing acts, or technical standards (RTS/ITS) drafted by the ESAs. Level 2 includes, '
            'for example, detailed procedural rules for applying Level 1 obligations; specifications of '
            'quantitative thresholds and benchmarks; technical formats and data standards; and timelines for '
            'compliance. Level 2 is designed to be flexible, allowing continuous technical adjustments without '
            'reopening primary legislation.'
        ),
    },
    {
        'key':  'L3',
        'name': 'Level 3 — Supervisory convergence',
        'label': 'level_3',
        'description': (
            'Level 3 consists of measures issued by committees of national supervisors — now transformed into '
            'the European supervisory authorities (EBA, ESMA, EIOPA) — responsible for advising the Commission '
            'on Level 1 and Level 2 acts and for issuing guidelines on the implementation of the rules. '
            'Level 3 includes guidelines, recommendations, opinions, Q&As, and supervisory convergence tools '
            'that do not have binding legal force but provide practical direction on how to apply the rules '
            'uniformly at national level.'
        ),
    },
    {
        'key':  'L4',
        'name': 'Level 4 — Enforcement',
        'label': 'level_4',
        'description': (
            'Level 4 concerns the enforcement and monitoring of compliance with EU rules by national governments '
            'and competent authorities, with a stronger role for the Commission in ensuring correct application. '
            'Level 4 includes infringement proceedings, enforcement actions, compliance checks, and peer reviews '
            'aimed at verifying that Member States are correctly implementing and applying Level 1 and Level 2 '
            'legislation.'
        ),
    },
]

LAMF_KEYS = [l['key']   for l in LAMFALUSSY_LEVELS]   # ['L1','L2','L3','L4']
LAMF_COLS = [f'lamf_{k}' for k in LAMF_KEYS]           # ['lamf_L1',...,'lamf_L4']
LABEL_TO_KEY = {l['label']: l['key'] for l in LAMFALUSSY_LEVELS}  # 'level_1' → 'L1'

print(f"Materia:    {MATERIA_NAME}")
print(f"Modello:    {LLM_MODEL}")
print(f"Livelli:    {LAMF_KEYS}")

Materia:    appalti_it
Modello:    gpt-4.1-mini
Livelli:    ['L1', 'L2', 'L3', 'L4']


## 2. Caricamento Dati e Segmentazione

In [4]:
# ── Carica IT ────────────────────────────────────────────────────────────────
nodes_it = pd.read_csv(INPUT_FILE)

if os.path.exists(INPUT_FILE_EU):
    nodes_eu_raw = pd.read_csv(INPUT_FILE_EU)
    # Normalizza colonne EU allo schema IT atteso da questo notebook
    nodes_eu = nodes_eu_raw.copy()
    if 'Id'    not in nodes_eu.columns: nodes_eu['Id']    = nodes_eu.get('celex', nodes_eu.get('label', ''))
    if 'Label' not in nodes_eu.columns: nodes_eu['Label'] = nodes_eu.get('celex', nodes_eu.get('label', ''))
    if 'text_status' not in nodes_eu.columns:
        nodes_eu['text_status'] = nodes_eu['segments'].apply(
            lambda s: 'ok' if (pd.notna(s) and str(s).strip() not in ('', '[]')) else 'no_text'
        )
    nodes = pd.concat([nodes_it, nodes_eu], ignore_index=True)
    print(f'IT: {len(nodes_it)} nodi  |  EU: {len(nodes_eu)} nodi  |  Totale: {len(nodes)}')
else:
    nodes = nodes_it
    print(f'File EU non trovato ({INPUT_FILE_EU}) — solo IT: {len(nodes)} nodi')

nodes_ok = nodes[nodes['text_status'] == 'ok'].copy()

print(f"Nodi totali: {len(nodes)}  |  ok: {len(nodes_ok)}")
print(nodes['text_status'].value_counts().to_string())

# ── Esplode segmenti in DataFrame flat ────────────────────────────────────────
rows = []
for _, node in nodes_ok.iterrows():
    celex = str(node.get('Label', node['Id']))
    title = str(node.get('title', ''))
    raw   = node.get('segments', '')
    if pd.isna(raw) or not str(raw).strip():
        continue
    try:
        segs = json.loads(str(raw))
    except (json.JSONDecodeError, ValueError):
        continue
    seen = set()
    for i, s in enumerate(segs):
        if s.get('tipo') != 'articolo':
            continue
        testo = str(s.get('testo', '')).strip()
        if len(testo) < 30:
            continue
        idf = str(s.get('identificatore', i))
        seg_id = f"{celex}__{idf}"
        if seg_id in seen:
            continue
        seen.add(seg_id)
        rows.append({
            'segment_id':    seg_id,
            'celex':         celex,
            'node_id':       str(node['Id']),
            'title_atto':    title,
            'tipo':          s.get('tipo'),
            'identificatore': idf,
            'testo':         testo,
        })

articles_df = pd.DataFrame(rows)

# ── Mappa celex → full_text (per il contesto documento) ──────────────────────
FULL_TEXTS = {}
for _, node in nodes_ok.iterrows():
    celex = str(node.get('Label', node['Id']))
    ft    = str(node.get('full_text', '') or '')
    if ft and ft != 'nan':
        FULL_TEXTS[celex] = ft

print(f"\nArticoli estratti: {len(articles_df):,}")
print(f"Atti coinvolti:    {articles_df['celex'].nunique():,}")
print(f"Full text disponibili: {len(FULL_TEXTS):,}")

IT: 29 nodi  |  EU: 3 nodi  |  Totale: 32
Nodi totali: 32  |  ok: 32
text_status
ok    32

Articoli estratti: 1,968
Atti coinvolti:    32
Full text disponibili: 32


## 3. Fase A — Classificazione Lamfalussy per Articolo (LLM)

Per ogni articolo:
1. Il testo viene tokenizzato (whitespace split) e ogni token riceve un indice 1-based
2. L'LLM segmenta il testo in **provisions** contigue e classifica ciascuna come
   `level_1`, `level_2`, `level_3`, `level_4` o `unassigned`
3. Le percentuali L1–L4 emergono dai **conteggi di token** per livello

Il documento integrale è fornito come contesto (`[FULL DOCUMENT]`) per permettere
all'LLM di interpretare ogni articolo nel suo contesto normativo.

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# PROMPT TEMPLATE — letto da file esterno
# Per modificare il prompt basta editare prompt.txt, senza toccare questo notebook.
# ─────────────────────────────────────────────────────────────────────────────
with open(PROMPT_FILE, encoding='utf-8') as _f:
    PROMPT_TEMPLATE = _f.read()

print(f'Prompt caricato da: {PROMPT_FILE}  ({len(PROMPT_TEMPLATE)} caratteri)')
# Verifica che i placeholder siano presenti
for ph in ['{{DOCUMENT_CONTEXT}}', '{{CHUNK_TEXT}}', '{{EXPECTED_UNITS}}', '{{DOCUMENT_NAME}}']:
    status = '✓' if ph in PROMPT_TEMPLATE else '✗ MANCANTE'
    print(f'  {status}  {ph}')


def tokenize_with_indices(text: str) -> tuple[list[str], str]:
    """Restituisce (words, testo_indicizzato)."""
    words = text.split()
    indexed = ' '.join(f'({i+1}){w}' for i, w in enumerate(words))
    return words, indexed


def build_prompt(doc_text: str, art_text: str, art_id: str, doc_name: str) -> str:
    """Costruisce il prompt per un singolo articolo."""
    _, indexed_art = tokenize_with_indices(art_text)
    chunk = f"[ARTICLE {art_id}]\n[ARTICLE_TEXT]\n{indexed_art}"
    unit_id   = 'i000001'
    expected  = json.dumps({unit_id: {'article_id': str(art_id), 'comma_id': None}}, indent=2)
    ctx       = doc_text[:DOC_CONTEXT_MAX_CHARS] if doc_text else '(not available)'
    return (PROMPT_TEMPLATE
        .replace('{{DOCUMENT_CONTEXT}}', ctx)
        .replace('{{CHUNK_TEXT}}',       chunk)
        .replace('{{EXPECTED_UNITS}}',   expected)
        .replace('{{DOCUMENT_NAME}}',    doc_name))


def parse_response(response_text: str, art_text: str) -> dict | None:
    """
    Parsa la risposta JSON dell'LLM e calcola le percentuali L1–L4
    dal conteggio dei token classificati.
    Restituisce None in caso di errore di parsing.
    """
    try:
        clean = re.sub(r'^```[a-z]*\n?', '', response_text.strip())
        clean = re.sub(r'\n?```$', '', clean)
        data  = json.loads(clean)
    except json.JSONDecodeError:
        return None

    classes = data.get('classifications', {})
    if not classes:
        return None
    unit = next(iter(classes.values()), {})
    provisions = unit.get('provisions', [])
    if not provisions:
        return None

    words, _ = tokenize_with_indices(art_text)
    total_tokens = len(words)

    # ── Conta token per livello ────────────────────────────────────────────────
    counts = {k: 0 for k in LAMF_KEYS}
    counts['unassigned'] = 0
    evidence = []

    for p in provisions:
        start  = int(p.get('start', 1))
        end    = int(p.get('end', total_tokens))
        label  = str(p.get('label', 'unassigned'))
        reason = str(p.get('reason', ''))
        span   = max(0, end - start + 1)
        key    = LABEL_TO_KEY.get(label, 'unassigned')
        counts[key] = counts.get(key, 0) + span

        # Estrai citazione verbatim dai token classificati
        if key != 'unassigned' and span > 0:
            quote = ' '.join(words[start-1:end])
            if len(quote) > 200:
                quote = quote[:197] + '...'
            evidence.append({'testo': quote, 'layer': key, 'motivo': reason})

    total_labeled = sum(counts.get(k, 0) for k in LAMF_KEYS)   # solo L1..L4
    if total_labeled == 0:
        return None

    pcts = {k: round(counts.get(k, 0) / total_labeled * 100, 2) for k in LAMF_KEYS}
    return {'pcts': pcts, 'evidence': evidence, 'provisions': provisions}


def call_llm(client, prompt: str) -> tuple[str, str]:
    """Chiama l'LLM con retry. Restituisce (response_text, status)."""
    for attempt in range(LLM_MAX_RETRIES):
        try:
            resp = client.chat.completions.create(
                model=LLM_MODEL,
                max_tokens=LLM_MAX_TOKENS,
                messages=[{'role': 'user', 'content': prompt}],
                temperature=0.0,
            )
            return resp.choices[0].message.content, 'ok'
        except openai.RateLimitError:
            time.sleep(LLM_RETRY_DELAY * (attempt + 1))
        except Exception as e:
            if attempt == LLM_MAX_RETRIES - 1:
                return str(e), 'error'
            time.sleep(LLM_RETRY_DELAY)
    return 'max_retries_exceeded', 'error'


print('Funzioni Fase A definite.')
print(f'Livelli Lamfalussy: {LAMF_KEYS}')

Prompt caricato da: ..\notebooks\prompt.txt  (7149 caratteri)
  ✓  {{DOCUMENT_CONTEXT}}
  ✓  {{CHUNK_TEXT}}
  ✓  {{EXPECTED_UNITS}}
  ✓  {{DOCUMENT_NAME}}
Funzioni Fase A definite.
Livelli Lamfalussy: ['L1', 'L2', 'L3', 'L4']


In [6]:
%%time
client = OpenAI()

# ── Gestione checkpoint ────────────────────────────────────────────────────────
done_ids = set()
if os.path.exists(SEGMENTS_LAMF_CKPT_FILE):
    ckpt_df  = pd.read_csv(SEGMENTS_LAMF_CKPT_FILE)
    done_ids = set(ckpt_df['segment_id'])
    print(f'Checkpoint: {len(done_ids):,} articoli già classificati.')
else:
    ckpt_df = pd.DataFrame()
    print('Nessun checkpoint — si parte da zero.')

todo_df = articles_df[~articles_df['segment_id'].isin(done_ids)].copy()
print(f'Articoli da classificare: {len(todo_df):,}  |  già ok: {len(done_ids):,}')

if len(todo_df) == 0:
    print('✓ Tutti i segmenti già classificati — si può passare alla Fase B.')

# ── Strutture condivise tra thread ────────────────────────────────────────────
ckpt_lock   = Lock()
buffer      = []
n_ok        = 0
n_error     = 0
n_processed = 0


def process_article(seg: pd.Series) -> dict:
    """Classifica un articolo e restituisce la riga da salvare."""
    celex  = seg['celex']
    art_id = seg['identificatore']
    testo  = seg['testo']
    prompt = build_prompt(
        doc_text  = FULL_TEXTS.get(celex, ''),
        art_text  = testo,
        art_id    = art_id,
        doc_name  = celex,
    )
    time.sleep(LLM_DELAY_SECONDS)
    response_text, status = call_llm(client, prompt)

    parsed = None
    if status == 'ok':
        parsed = parse_response(response_text, testo)
        if parsed is None:
            status = 'parse_error'

    row = {
        'segment_id':    seg['segment_id'],
        'celex':         celex,
        'node_id':       seg['node_id'],
        'tipo':          seg['tipo'],
        'identificatore': art_id,
        'llm_status':    status,
        'evidence':      json.dumps(parsed['evidence'], ensure_ascii=False) if parsed else '[]',
        'provisions_raw': json.dumps(parsed['provisions'], ensure_ascii=False) if parsed else '[]',
    }
    for col, key in zip(LAMF_COLS, LAMF_KEYS):
        row[col] = round(parsed['pcts'][key], 2) if parsed else 0.0
    return row


def save_checkpoint(new_rows):
    if os.path.exists(SEGMENTS_LAMF_CKPT_FILE):
        existing = pd.read_csv(SEGMENTS_LAMF_CKPT_FILE)
    else:
        existing = pd.DataFrame()
    parts    = [p for p in [existing, pd.DataFrame(new_rows)] if not p.empty]
    combined = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    combined.to_csv(SEGMENTS_LAMF_CKPT_FILE, index=False)


# ── Loop parallelo ─────────────────────────────────────────────────────────────
total = len(todo_df)
todo_records = [row for _, row in todo_df.iterrows()]

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(process_article, seg): seg['segment_id'] for seg in todo_records}
    for future in as_completed(futures):
        row = future.result()
        with ckpt_lock:
            buffer.append(row)
            n_processed += 1
            if row['llm_status'] == 'ok':
                n_ok += 1
            else:
                n_error += 1
            if n_processed % CHECKPOINT_EVERY == 0 or n_processed == total:
                save_checkpoint(buffer)
                buffer.clear()
                pct = n_processed / total * 100 if total else 100
                print(f'  [{n_processed:>5}/{total}]  {pct:5.1f}%   ok: {n_ok}   errori: {n_error}')

print()
print('=' * 50)
print(f'FASE A — ok: {n_ok:,}   errori: {n_error:,}')
print('=' * 50)

Checkpoint: 1,968 articoli già classificati.
Articoli da classificare: 0  |  già ok: 1,968
✓ Tutti i segmenti già classificati — si può passare alla Fase B.

FASE A — ok: 0   errori: 0
CPU times: total: 375 ms
Wall time: 381 ms


In [7]:
# ── Salva output finale Fase A ─────────────────────────────────────────────────
segments_lamf = pd.read_csv(SEGMENTS_LAMF_CKPT_FILE)
segments_lamf.to_csv(SEGMENTS_LAMF_FILE, index=False)

ok_mask = segments_lamf['llm_status'] == 'ok'
print(f'Salvato: {SEGMENTS_LAMF_FILE}')
print(f'Totale: {len(segments_lamf):,}  |  ok: {ok_mask.sum():,}')
print()
print('Distribuzione media % per livello Lamfalussy (articoli ok):')
for col, key in zip(LAMF_COLS, LAMF_KEYS):
    level    = next(l for l in LAMFALUSSY_LEVELS if l['key'] == key)
    mean_pct = segments_lamf.loc[ok_mask, col].mean()
    bar      = '█' * int(mean_pct / 2)
    print(f"  {level['name'][:45]:.<46} {mean_pct:5.1f}%  {bar}")

Salvato: ..\data\output\appalti_it\segments_lamfalussy.csv
Totale: 1,968  |  ok: 1,846

Distribuzione media % per livello Lamfalussy (articoli ok):
  Level 1 — Framework principles................  50.4%  █████████████████████████
  Level 2 — Operational rules...................  41.3%  ████████████████████
  Level 3 — Supervisory convergence.............   2.9%  █
  Level 4 — Enforcement.........................   5.3%  ██


## 4. Fase B — Entropia per Articolo e per Atto

Per ogni articolo: **entropia di Shannon normalizzata** sulla distribuzione L1–L4.

$$H(a) = -\frac{\sum_{l} p_{al} \log_2(p_{al} + \varepsilon)}{\log_2(4)}$$

Zero = articolo monofunzionale. Uno = distribuzione uniforme sui 4 livelli.

In [8]:
def entropy_norm(row, lamf_cols):
    eps   = 1e-9
    probs = np.array([float(row.get(c, 0)) for c in lamf_cols]) / 100.0
    probs = np.clip(probs, 0, 1)
    s     = probs.sum()
    if s < eps:
        return 0.0
    probs = probs / s
    rawH  = -np.sum(probs * np.log2(probs + eps))
    maxH  = math.log2(len(lamf_cols)) if len(lamf_cols) > 1 else 1.0
    return float(np.clip(rawH / maxH, 0, 1))


# Ricarica dal file finale (compatibile con run parziali)
seg_df  = pd.read_csv(SEGMENTS_LAMF_FILE)
art_df  = seg_df[
    (seg_df['llm_status'] == 'ok') &
    (seg_df['segment_id'].isin(set(articles_df['segment_id'])))
].copy()

for col in LAMF_COLS:
    if col not in art_df.columns:
        art_df[col] = 0.0

art_df['entropy'] = art_df.apply(lambda r: entropy_norm(r, LAMF_COLS), axis=1)
art_df['dominant_lamf'] = art_df[LAMF_COLS].idxmax(axis=1).str.replace('lamf_', '', regex=False)
art_df['articolo_id']   = art_df['identificatore']

art_df.to_csv(NODES_LAMFALUSSY_FILE, index=False)

print(f'Salvato: {NODES_LAMFALUSSY_FILE}')
print(f'Articoli: {len(art_df):,}  |  Atti: {art_df["celex"].nunique():,}')
print(f'Entropia media: {art_df["entropy"].mean():.4f}  |  max: {art_df["entropy"].max():.4f}')
print()
print('Distribuzione media % per livello Lamfalussy (articoli ok):')
for col, key in zip(LAMF_COLS, LAMF_KEYS):
    level    = next(l for l in LAMFALUSSY_LEVELS if l['key'] == key)
    mean_pct = art_df[col].mean()
    bar      = '█' * int(mean_pct / 2)
    print(f"  {level['name'][:45]:.<46} {mean_pct:5.1f}%  {bar}")

Salvato: ..\data\output\appalti_it\nodes_lamfalussy.csv
Articoli: 1,846  |  Atti: 32
Entropia media: 0.1226  |  max: 0.9103

Distribuzione media % per livello Lamfalussy (articoli ok):
  Level 1 — Framework principles................  50.4%  █████████████████████████
  Level 2 — Operational rules...................  41.3%  ████████████████████
  Level 3 — Supervisory convergence.............   2.9%  █
  Level 4 — Enforcement.........................   5.3%  ██


## 5. Fase C — Score di Ibridità per Atto

Lo score di ibridità di un atto è l'**entropia della distribuzione L1-L4 media** dei suoi articoli.
Un atto è **puro** se la sua distribuzione aggregata è concentrata su un solo livello (H ≈ 0).
Un atto è **ibrido** se la sua distribuzione aggregata è equamente spalmata sui livelli (H → 1).

> **Nota:** questa misura differisce dalla media delle entropie per articolo.
> Un atto con metà articoli puri L1 e metà puri L2 ottiene H_atto ≈ 0.5 (corretto),
> mentre la media delle entropie per articolo darebbe ≈ 0 (sbagliato).

In [9]:
LAMF_NAMES = {l['key']: l['name'] for l in LAMFALUSSY_LEVELS}

def entropy_of_mean(group, lamf_cols):
    """Entropy of the mean L1-L4 distribution across all articles in the group.
    This captures act-level hybridness: an act where half articles are pure L1
    and half are pure L2 correctly gets high entropy, unlike mean-of-entropies."""
    eps = 1e-9
    mean_vals = group[lamf_cols].mean().values.astype(float)
    s = mean_vals.sum()
    if s < eps:
        return 0.0
    probs = mean_vals / s
    rawH = -np.sum(probs * np.log2(probs + eps))
    maxH = math.log2(len(lamf_cols))
    return float(np.clip(rawH / maxH, 0, 1))


def agg_atto(group):
    dom = group['dominant_lamf'].mode()
    hyb_max_row = group.nlargest(1, 'entropy')
    dominant_key = dom.iloc[0] if len(dom) else ''
    return pd.Series({
        'hybridity_score':     entropy_of_mean(group, LAMF_COLS),
        'hybridity_score_old': group['entropy'].mean(),   # kept for reference, can be dropped later
        'hybridity_std':       group['entropy'].std(),
        'hybridity_max':       group['entropy'].max(),
        'n_articles':          len(group),
        'dominant_lamf':       dominant_key,
        'dom':                 LAMF_NAMES.get(dominant_key, dominant_key),
        'dominant_lamf_pct':   (group['dominant_lamf'] == dominant_key).mean() * 100 if dominant_key else 0.0,
        'most_hybrid_article': hyb_max_row['articolo_id'].iloc[0] if len(hyb_max_row) else '',
        **{col: group[col].mean() for col in LAMF_COLS},
    })


hybridity_df = art_df.groupby('celex').apply(agg_atto).reset_index()

# Aggiunge metadati dal nodo originale
meta_cols  = [c for c in ['Id', 'Label', 'title'] if c in nodes.columns]
nodes_meta = nodes[meta_cols].copy()
if 'Label' in nodes_meta.columns:
    nodes_meta = nodes_meta.rename(columns={'Label': 'celex'})

hybridity_df = (hybridity_df
    .merge(nodes_meta, on='celex', how='left')
    .drop_duplicates(subset=['celex'], keep='first')
    .sort_values('hybridity_score', ascending=False))

hybridity_df = hybridity_df.loc[:, ~hybridity_df.columns.duplicated()]
hybridity_df.to_csv(NODES_HYBRIDITY_FILE, index=False)

print(f'Salvato: {NODES_HYBRIDITY_FILE}')
print(f'Atti analizzati: {len(hybridity_df):,}')
desc = hybridity_df['hybridity_score'].describe()
print(f"\nStatistiche hybridity_score (Lamfalussy 4 livelli):")
print(f"  Media:   {desc['mean']:.4f}")
print(f"  Mediana: {desc['50%']:.4f}")
print(f"  Max:     {desc['max']:.4f}")
print(f"  Std:     {desc['std']:.4f}")
print()
print('Top 5 atti più ibridi:')
for _, row in hybridity_df.head(5).iterrows():
    celex_label = row.get('Label', row.get('celex', ''))
    print(f"  {celex_label:<22}  score={row['hybridity_score']:.3f}  dominant={row['dominant_lamf']}")

# Verifica CSV salvato
_check = pd.read_csv(NODES_HYBRIDITY_FILE)
print(f"\nVerifica CSV — colonne: {_check.columns.tolist()}")
print(_check.dtypes.to_string())
print(_check.head(3).to_string())

Salvato: ..\data\output\appalti_it\nodes_hybridity.csv
Atti analizzati: 32

Statistiche hybridity_score (Lamfalussy 4 livelli):
  Media:   0.4252
  Mediana: 0.5653
  Max:     0.7427
  Std:     0.2788

Top 5 atti più ibridi:
  dlgs_59_2010            score=0.743  dominant=L2
  l_90_2024               score=0.724  dominant=L2
  dlgs_228_2011           score=0.705  dominant=L2
  dlgs_33_2013            score=0.688  dominant=L2
  l_190_2012              score=0.676  dominant=L1

Verifica CSV — colonne: ['celex', 'hybridity_score', 'hybridity_score_old', 'hybridity_std', 'hybridity_max', 'n_articles', 'dominant_lamf', 'dom', 'dominant_lamf_pct', 'most_hybrid_article', 'lamf_L1', 'lamf_L2', 'lamf_L3', 'lamf_L4', 'Id']
celex                      str
hybridity_score        float64
hybridity_score_old    float64
hybridity_std          float64
hybridity_max          float64
n_articles               int64
dominant_lamf              str
dom                        str
dominant_lamf_pct      float64

## 6. Fase D — Export heatmaps.json + Patch HTML

Costruisce `heatmaps.json` con le percentuali L1–L4 per ogni articolo,
poi patcha l'HTML per aggiornare tutte le referenze da 2 a 4 livelli Lamfalussy.

In [10]:
import json as _json
import re as _re
import os as _os

# ── 1. Costruisci HEATMAPS dict ───────────────────────────────────────────────
# Mappa celex → testi articoli (per il campo 'txt' nell'heatmap)
ART_TEXTS = {}
if _os.path.exists(INPUT_FILE):
    texts_df = pd.read_csv(INPUT_FILE)
    id_col   = 'Id' if 'Id' in texts_df.columns else 'celex'
    for _, row in texts_df.iterrows():
        celex = str(row[id_col])
        raw   = row.get('segments', '')
        if not raw or str(raw) in ('nan', '[]', ''):
            continue
        try:
            for seg in _json.loads(str(raw)):
                if seg.get('tipo') == 'articolo':
                    ART_TEXTS[(celex, str(seg.get('identificatore', '')))] = seg.get('testo', '')
        except Exception:
            pass
print(f'ART_TEXTS: {len(ART_TEXTS)} articoli')

# Mappa segment_id → evidence (da segments_lamfalussy.csv)
EVIDENCE = {}
seg_file = pd.read_csv(SEGMENTS_LAMF_FILE)
for _, row in seg_file[seg_file['llm_status'] == 'ok'].iterrows():
    celex = str(row['celex'])
    idf   = str(row['identificatore'])
    raw   = row.get('evidence', '[]')
    try:
        EVIDENCE[(celex, idf)] = _json.loads(raw) if isinstance(raw, str) else []
    except Exception:
        EVIDENCE[(celex, idf)] = []
print(f'EVIDENCE:  {len(EVIDENCE)} articoli con evidence')

# Costruisce HEATMAPS
HEATMAPS = {}
lamf_ok  = pd.read_csv(NODES_LAMFALUSSY_FILE)

for celex, grp in lamf_ok.groupby('celex'):
    arts = []
    for _, row in grp.sort_values('identificatore', key=lambda s: s.apply(
        lambda x: int(_re.match(r'(\d+)', str(x)).group(1)) if _re.match(r'\d', str(x)) else 9999
    )).iterrows():
        idf   = str(row['articolo_id'])
        vals  = [round(float(row.get(c, 0)), 2) for c in LAMF_COLS]
        total = sum(vals)
        if total > 0 and abs(total - 100) > 0.5:
            vals = [round(v / total * 100, 2) for v in vals]
        H_val = round(float(row.get('entropy', 0)), 4)
        lamf  = {k: vals[i] for i, k in enumerate(LAMF_KEYS)}
        arts.append({
            'id':     idf,
            'vals':   vals,           # [L1%, L2%, L3%, L4%]
            'H':      H_val,          # entropia articolo (layer emersi = lamfalussy)
            'lamf':   lamf,           # {L1: %, L2: %, L3: %, L4: %}
            'H_lamf': H_val,          # identico a H (mantenuto per compat. HTML)
            'ev':     EVIDENCE.get((celex, idf), []),
            'txt':    ART_TEXTS.get((celex, idf), ''),
        })
    if arts:
        HEATMAPS[celex] = arts

print(f'HEATMAPS:  {len(HEATMAPS)} atti  |  {sum(len(v) for v in HEATMAPS.values())} articoli totali')

# Salva heatmaps.json accanto all'HTML
html_dir  = _os.path.dirname(HTML_FILE)
json_path = _os.path.join(html_dir, 'heatmaps.json')
with open(json_path, 'w', encoding='utf-8') as f:
    _json.dump(HEATMAPS, f, ensure_ascii=False)
print(f'\n✓ Salvato {json_path}  ({_os.path.getsize(json_path)//1024} KB)')

ART_TEXTS: 1709 articoli
EVIDENCE:  1846 articoli con evidence
HEATMAPS:  32 atti  |  1846 articoli totali

✓ Salvato ..\heatmaps.json  (4737 KB)


In [11]:
# ── Estrazione archi da citazioni testuali ─────────────────────────────────────
# Patterna sui riferimenti normativi italiani nel testo degli articoli
# e li risolve contro i nodi del catalogo tramite anno+numero.

import re as _re_cit

EDGES_FILE_IT = os.path.join(output_path, 'edges_it_internal.csv')

# Mappa (tipo_normalizzato, anno, numero) → slug
# costruita dai metadati seed in nodes_texts_it.csv
nodes_it_df = pd.read_csv(INPUT_FILE)
id_col_it   = 'Id' if 'Id' in nodes_it_df.columns else 'id'

TIPO_ALIASES = {
    'decreto legislativo':                     'dlgs',
    'decreto-legislativo':                     'dlgs',
    'd.lgs':                                   'dlgs',
    'd.lgs.':                                  'dlgs',
    'dlgs':                                    'dlgs',
    'legge':                                   'l',
    'l.':                                      'l',
    'decreto del presidente della repubblica': 'dpr',
    'decreto.del.presidente.della.repubblica': 'dpr',
    'd.p.r':                                   'dpr',
    'd.p.r.':                                  'dpr',
    'dpr':                                     'dpr',
    'decreto-legge':                           'dl',
    'decreto legge':                           'dl',
    'd.l.':                                    'dl',
    'dl':                                      'dl',
    'decreto ministeriale':                    'dm',
    'decreto del ministro':                    'dm',
    'd.m.':                                    'dm',
    'dm':                                      'dm',
}

# Pattern: cattura tipo, data/anno, numero
# Es: "decreto legislativo 18 aprile 2016, n. 50"
# Es: "legge 21 giugno 2022, n. 78"
# Es: "decreto-legge 16 luglio 2020, n. 76"
MESI = r'(?:gennaio|febbraio|marzo|aprile|maggio|giugno|luglio|agosto|settembre|ottobre|novembre|dicembre)'
PAT_FULL = _re_cit.compile(
    r'(decreto(?:\s+del\s+presidente\s+della\s+repubblica|[\s\-]?legislativo|[\s\-]?legge|[\s\s]ministeriale)?'
    r'|legge|l\.)\s+'
    r'(?:\d{1,2}\s+' + MESI + r'\s+)?'
    r'(\d{4})\s*,?\s*n\.?\s*(\d+)',
    _re_cit.IGNORECASE
)
# Pattern più semplice per catturare anche "n. 76 del 2020" o "n. 50/2016"
PAT_SHORT = _re_cit.compile(
    r'(d\.lgs\.|d\.p\.r\.|d\.l\.|d\.m\.|legge)\s*'
    r'(?:\d{1,2}\s+' + MESI + r'\s+)?'
    r'(\d{4})\s*[,\s]+n\.?\s*(\d+)',
    _re_cit.IGNORECASE
)

# ── Pattern per tipo di relazione ─────────────────────────────────────────────
# Finestra di contesto: 120 caratteri prima del riferimento normativo
WINDOW = 120

def classify_relation(context):
    """Classifica il tipo di relazione dal contesto testuale prima della citazione."""
    ctx = context.lower()

    # REPEALS — abrogazione
    if any(p in ctx for p in [
        'abrogat', 'abrogazion',
        'cessa di avere efficacia', 'cessano di avere efficacia',
        'cessa di applicarsi', 'cessano di applicarsi',
        'è soppresso', 'sono soppressi', 'è soppressa', 'sono soppresse',
        'è abrogato', 'sono abrogati', 'è abrogata', 'sono abrogate',
        'viene abrogat', 'vengono abrogat',
        'non si applica più', 'perde efficacia',
    ]):
        return 'REPEALS', 'mod'

    # AMENDS — modifica
    if any(p in ctx for p in [
        'modificat', 'sostituit',
        'è sostituito', 'sono sostituiti', 'è sostituita', 'sono sostituite',
        'è inserito', 'sono inseriti', 'è inserita', 'sono inserite',
        'è aggiunto', 'sono aggiunti', 'è aggiunta', 'sono aggiunte',
        'le parole', 'dopo le parole', 'dopo il comma', "dopo l'articolo",
        'è aggiornato', 'novellat', 'è integrato', 'sono integrati',
        'è così modificat',
    ]):
        return 'AMENDS', 'mod'

    # BASED_ON — attuazione/delega
    if any(p in ctx for p in [
        'in attuazione', 'in applicazione', 'ai sensi',
        'in conformità', 'in esecuzione', 'delegat',
        'su delega', 'recepimento', 'recepisce'
    ]):
        return 'BASED_ON', 'hier'

    # DEROGATES
    if any(p in ctx for p in [
        'deroga', 'in deroga', 'non si applica'
    ]):
        return 'DEROGATES', 'proc'

    # Default
    return 'CITES', 'ref'

def normalize_tipo(raw):
    raw = raw.strip().lower().rstrip('.')
    for alias, norm in TIPO_ALIASES.items():
        if raw.startswith(alias):
            return norm
    return None

# Costruisce lookup: (tipo_norm, anno, numero) → slug
# + mappa slug → anno
slug_lookup = {}
slug_to_year = {}

for _, row in nodes_it_df.iterrows():
    slug = str(row[id_col_it])
    parts = slug.split('_')

    if len(parts) >= 3:
        tipo  = parts[0]
        num   = parts[1]
        anno  = parts[2]

        # lookup principale
        slug_lookup[(tipo, anno, num)] = slug

        # mappa anno (con controllo robusto)
        try:
            slug_to_year[slug] = int(anno)
        except ValueError:
            pass

    # Anche da colonne esplicite se esistono
    for col in ['anno', 'numero', 'tipo']:
        pass  # già estratto dallo slug

print(f'Lookup nodi: {len(slug_lookup)} voci')
print(list(slug_lookup.items())[:5])

# Scansiona tutti i testi
text_edges = []
seen_text  = set()

for _, node_row in nodes_it_df.iterrows():
    src_slug = str(node_row[id_col_it])
    segs_raw = node_row.get('segments', '')
    full_txt = str(node_row.get('full_text', '') or '')

    texts_to_scan = [full_txt]
    if segs_raw and str(segs_raw) not in ('nan', '[]', ''):
        try:
            for seg in _json.loads(str(segs_raw)):
                texts_to_scan.append(str(seg.get('testo', '')))
        except Exception:
            pass

    combined = ' '.join(texts_to_scan)

    for pat in [PAT_FULL, PAT_SHORT]:
        for m in pat.finditer(combined):
            tipo_raw = m.group(1)
            anno     = m.group(2)
            numero   = m.group(3)
            tipo_n   = normalize_tipo(tipo_raw)
            if not tipo_n:
                continue
            dst_slug = slug_lookup.get((tipo_n, anno, numero))
            if not dst_slug or dst_slug == src_slug:
                continue
            key = (src_slug, dst_slug)
            if key in seen_text:
                continue
            seen_text.add(key)
            # Classifica la relazione dal contesto testuale
            start   = max(0, m.start() - WINDOW)
            end     = min(len(combined), m.end() + WINDOW)
            context = combined[start:end]
            etype, family = classify_relation(context)
            text_edges.append({
                'src_slug': src_slug,
                'dst_slug': dst_slug,
                'type':     etype,
                'family':   family,
                'w':        2 if etype in ('REPEALS','AMENDS','BASED_ON') else 1,
            })

print(f'\nArchi estratti dai testi: {len(text_edges)}')
df_text_edges = pd.DataFrame(text_edges)

if len(df_text_edges):
    df_text_edges.to_csv(EDGES_FILE_IT, index=False)
    print(f'Salvato: {EDGES_FILE_IT}  ({len(df_text_edges)} archi)')

if len(df_text_edges):
    print(df_text_edges['src_slug'].value_counts().head(15).to_string())
    # Verifica l_120_2020
    mask = (df_text_edges['src_slug']=='l_120_2020')|(df_text_edges['dst_slug']=='l_120_2020')
    print(f'\nArchi l_120_2020: {mask.sum()}')
    print(df_text_edges[mask].to_string())

# Verifica CSV salvato
_echeck = pd.read_csv(EDGES_FILE_IT)
print(f'\nVerifica CSV — colonne: {_echeck.columns.tolist()}')
print(f'Righe: {len(_echeck)}')
print(_echeck.head(3).to_string())

Lookup nodi: 29 voci
[(('dlgs', '2023', '36'), 'dlgs_36_2023'), (('dlgs', '2016', '50'), 'dlgs_50_2016'), (('dlgs', '2006', '163'), 'dlgs_163_2006'), (('dlgs', '2024', '209'), 'dlgs_209_2024'), (('dlgs', '2017', '56'), 'dlgs_56_2017')]

Archi estratti dai testi: 83
Salvato: ..\data\output\appalti_it\edges_it_internal.csv  (83 archi)
src_slug
dlgs_36_2023     17
dlgs_56_2017      9
dl_13_2023        9
dlgs_209_2024     7
dlgs_33_2013      6
dlgs_97_2016      6
l_190_2012        5
dlgs_228_2011     4
dlgs_229_2011     3
dlgs_163_2006     2
dpr_207_2010      2
dlgs_218_2012     2
dlgs_59_2010      2
dl_135_2018       2
dlgs_50_2016      1

Archi l_120_2020: 3
         src_slug    dst_slug    type family  w
10   dlgs_36_2023  l_120_2020  AMENDS    mod  2
25  dlgs_209_2024  l_120_2020   CITES    ref  1
77     dl_13_2023  l_120_2020   CITES    ref  1

Verifica CSV — colonne: ['src_slug', 'dst_slug', 'type', 'family', 'w']
Righe: 83
       src_slug      dst_slug   type family  w
0  dlgs_36_20

## 7. Fase E — Entropia Globale via Diffusione di Citazioni

L'entropia locale `H` calcolata in Fase B misura solo l'**ibridità interna** di un atto:
quanto i suoi articoli sono distribuiti sui 4 livelli Lamfalussy. Non cattura la
**contaminazione funzionale via citazioni**: un atto monofunzionale che cita atti
ibridi eredita complessità nel sistema in cui opera.

Implementiamo la **diffusione del profilo di livello**:

$$
\mathbf{p}_{\text{global}}(v) = (1-\lambda)\,\mathbf{p}_{\text{local}}(v) + \lambda \sum_{u} \tilde{w}(v,u)\,\mathbf{p}_{\text{global}}(u)
$$

In forma chiusa: $\mathbf{P}_{\text{glob}} = (1-\lambda)(I - \lambda \tilde{W})^{-1} \mathbf{P}_{\text{loc}}$

con $\tilde{W}$ riga-stocastica costruita dagli archi del grafo. Gli archi
`IMPLEMENTED_BY` (EU→IT) sono **invertiti** nel calcolo perché semanticamente
l'atto IT eredita complessità interpretativa dalla direttiva EU che recepisce.

Definiamo tre quantità:
- `H_local(v)` = Shannon di $\mathbf{p}_{\text{local}}(v)$ — già calcolata in Fase B
- `H_global(v)` = Shannon di $\mathbf{p}_{\text{global}}(v)$
- `Δ(v) = H_global(v) − H_local(v)` = **deriva di contaminazione**

Δ > 0 → l'atto sembra coeso ma partecipa a un ecosistema ibrido.  
Δ < 0 → cita atti più ordinati di sé.


In [12]:
import numpy as _np_diff
import json as _json_diff
import re as _re_diff

LAMBDA = 0.5  # peso delle citazioni nella diffusione

# ── 1. Rilegge NODES e EDGES dall'HTML appena patchato ────────────────────────
with open(HTML_FILE, 'r', encoding='utf-8') as f:
    _html_d = f.read()
_nodes_d = _json_diff.loads(_re_diff.search(r'const NODES\s*=\s*(\[.*?\]);', _html_d, _re_diff.DOTALL).group(1))
_edges_d = _json_diff.loads(_re_diff.search(r'const EDGES\s*=\s*(\[.*?\]);', _html_d, _re_diff.DOTALL).group(1))

# ── 2. Costruisce vettore p_local e mappa idx ────────────────────────────────
NODE_IDS = [n['id'] for n in _nodes_d]
N        = len(NODE_IDS)
idx      = {nid: i for i, nid in enumerate(NODE_IDS)}

P_loc = _np_diff.zeros((N, 4))
for n in _nodes_d:
    i  = idx[n['id']]
    lf = n.get('lamf', {})
    for j, k in enumerate(LAMF_KEYS):
        P_loc[i, j] = float(lf.get(k, 0.0))

# Normalizza per riga (somma = 100% → 1)
row_sum_loc = P_loc.sum(axis=1, keepdims=True)
row_sum_loc[row_sum_loc == 0] = 1.0
P_loc_norm  = P_loc / row_sum_loc

# ── 3. Costruisce W (pesi grezzi). Flip degli archi `recep` (EU→IT diventa IT→EU)
W = _np_diff.zeros((N, N))
n_flipped = 0
for e in _edges_d:
    s, t, fam, w = e['s'], e['t'], e['family'], float(e.get('w', 1))
    if s not in idx or t not in idx:
        continue
    if fam == 'recep':
        s, t = t, s          # IT eredita da EU
        n_flipped += 1
    W[idx[s], idx[t]] += w

# ── 4. Row-stochastic W̃ ────────────────────────────────────────────────────
row_sum_W = W.sum(axis=1, keepdims=True)
row_sum_W[row_sum_W == 0] = 1.0
W_tilde   = W / row_sum_W

# ── 5. Risoluzione chiusa: P_glob = (1-λ)(I - λ W̃)⁻¹ P_loc ──────────────────
I_mat  = _np_diff.eye(N)
P_glob = (1 - LAMBDA) * _np_diff.linalg.solve(I_mat - LAMBDA * W_tilde, P_loc_norm)

# Ri-normalizza per riga (per stabilità numerica)
ps = P_glob.sum(axis=1, keepdims=True)
ps[ps == 0] = 1.0
P_glob = P_glob / ps

# ── 6. Entropia normalizzata di ogni distribuzione ───────────────────────────
def _shannon4(p, eps=1e-9):
    p = _np_diff.clip(p, 0, 1)
    s = p.sum()
    if s < eps:
        return 0.0
    p = p / s
    H = -_np_diff.sum(p * _np_diff.log2(p + eps))
    return float(_np_diff.clip(H / _np_diff.log2(len(p)), 0, 1))

H_loc_arr  = _np_diff.array([_shannon4(P_loc_norm[i]) for i in range(N)])
H_glob_arr = _np_diff.array([_shannon4(P_glob[i])     for i in range(N)])
H_delta    = H_glob_arr - H_loc_arr

# ── 7. Aggiorna NODES con campi globali ──────────────────────────────────────
for n in _nodes_d:
    i = idx[n['id']]
    n['H_local']    = round(float(H_loc_arr[i]),  3)
    n['H_global']   = round(float(H_glob_arr[i]), 3)
    n['H_delta']    = round(float(H_delta[i]),    3)
    n['lamf_loc']   = {k: round(float(P_loc_norm[i, j] * 100), 1) for j, k in enumerate(LAMF_KEYS)}
    n['lamf_glob']  = {k: round(float(P_glob[i, j]    * 100), 1) for j, k in enumerate(LAMF_KEYS)}
    # Il campo 'H' originale (= H_local) è lasciato invariato per retrocompatibilità

# ── 8. Riscrive NODES nell'HTML ──────────────────────────────────────────────
nodes_str_new = 'const NODES  = ' + _json_diff.dumps(_nodes_d, ensure_ascii=False) + ';'
_html_d       = _re_diff.sub(r'const NODES\s*=\s*\[.*?\];', lambda m: nodes_str_new, _html_d, flags=_re_diff.DOTALL)
with open(HTML_FILE, 'w', encoding='utf-8') as f:
    f.write(_html_d)

# ── 9. Diagnostica ───────────────────────────────────────────────────────────
print(f'λ = {LAMBDA}')
print(f'Nodi: {N}  |  Archi: {len(_edges_d)}  |  Archi recep flippati: {n_flipped}')
print()
print(f'H_local:   mean={H_loc_arr.mean():.3f}  max={H_loc_arr.max():.3f}')
print(f'H_global:  mean={H_glob_arr.mean():.3f}  max={H_glob_arr.max():.3f}')
print(f'Δ medio:    {H_delta.mean():+.3f}')
print()
print('Top 5 atti per Δ (più contaminati via citazioni):')
order = _np_diff.argsort(-H_delta)[:5]
for i in order:
    nid = NODE_IDS[i]
    print(f'  {nid:<22}  H_loc={H_loc_arr[i]:.3f}  H_glob={H_glob_arr[i]:.3f}  Δ={H_delta[i]:+.3f}')

print()
print('Top 5 atti per H_global (più ibridi nel sistema):')
order = _np_diff.argsort(-H_glob_arr)[:5]
for i in order:
    nid = NODE_IDS[i]
    print(f'  {nid:<22}  H_loc={H_loc_arr[i]:.3f}  H_glob={H_glob_arr[i]:.3f}  Δ={H_delta[i]:+.3f}')

print(f'\n✓ HTML ri-patchato con campi: H_local, H_global, H_delta, lamf_loc, lamf_glob')


λ = 0.5
Nodi: 32  |  Archi: 100  |  Archi recep flippati: 6

H_local:   mean=0.425  max=0.742
H_global:  mean=0.546  max=0.826
Δ medio:    +0.121

Top 5 atti per Δ (più contaminati via citazioni):
  l_120_2020              H_loc=0.000  H_glob=0.431  Δ=+0.431
  l_11_2016               H_loc=0.000  H_glob=0.396  Δ=+0.396
  l_114_2014              H_loc=0.000  H_glob=0.386  Δ=+0.386
  dlgs_50_2016            H_loc=0.142  H_glob=0.463  Δ=+0.321
  dlgs_56_2017            H_loc=0.321  H_glob=0.633  Δ=+0.312

Top 5 atti per H_global (più ibridi nel sistema):
  l_108_2021              H_loc=0.613  H_glob=0.826  Δ=+0.213
  dlgs_59_2010            H_loc=0.742  H_glob=0.776  Δ=+0.034
  dlgs_228_2011           H_loc=0.705  H_glob=0.773  Δ=+0.069
  l_136_2010              H_loc=0.617  H_glob=0.767  Δ=+0.150
  dlgs_229_2011           H_loc=0.623  H_glob=0.737  Δ=+0.114

✓ HTML ri-patchato con campi: H_local, H_global, H_delta, lamf_loc, lamf_glob
